# Clinical Data Validation Mock data auditing-- creating mock data using python and SQL to audit it

In [7]:
import pandas as pd
import sqlite3

# 1. Create a mock healthcare dataset
data = {
    "patient_id": [101, 102, 103, 104, 105, 106],
    "patient_name": [
        "Alice Smith",
        "Bob Jones",
        "Charlie Brown",
        "David Wilson",
        "Eva Davis",
        "Frank Miller",
    ],
    "systolic_bp": [
        120,
        140,
        999,
        115,
        130,
        -50,
    ],  # 999 and -50 are clearly data errors!
    "diastolic_bp": [80, 90, 60, 75, 85, 70],
    "heart_rate": [72, 85, 110, 65, 400, 78],  # 400 is physically impossible!
}

df = pd.DataFrame(data)

# 2. Connect to a temporary in-memory SQL database
conn = sqlite3.connect(":memory:")

# 3. Push our mock data into a SQL table called 'vitals'
df.to_sql("vitals", conn, index=False, if_exists="replace")

print("Database created and populated successfully!")

Database created and populated successfully!


# Query 1: Hunting for Impossible Vitals

In [ ]:
# Function to run our SQL queries and return a clean Pandas DataFrame
def run_audit(query):
    return pd.read_sql_query(query, conn)


# Query 1: Find physically impossible vitals (Validation check)
query_impossible_vitals = """
SELECT * FROM vitals
WHERE systolic_bp < 0 
   OR systolic_bp > 300
   OR heart_rate > 300;
"""

print("--- FAILED VALIDATION: Impossible Vitals Detected ---")
display(run_audit(query_impossible_vitals))

--- FAILED VALIDATION: Impossible Vitals Detected ---


,patient_id,patient_name,systolic_bp,diastolic_bp,heart_rate
0,103,Charlie Brown,999,60,110
1,105,Eva Davis,130,85,400
2,106,Frank Miller,-50,70,78


# Query 2: Identifying High-Risk Patients.

In [ ]:
# Query 2: Find patients needing immediate clinical attention
query_hypertension = """
SELECT patient_id, patient_name, systolic_bp, diastolic_bp
FROM vitals
WHERE (systolic_bp >= 140 OR diastolic_bp >= 90)
  AND systolic_bp < 300; -- Exclude the obvious data errors we found earlier!
"""

print("\n--- CLINICAL ALERT: Patients with Stage 2 Hypertension ---")
display(run_audit(query_hypertension))


--- CLINICAL ALERT: Patients with Stage 2 Hypertension ---


,patient_id,patient_name,systolic_bp,diastolic_bp
0,102,Bob Jones,140,90
